Lead IQ — Feature Store setup

Registers Lead IQ's features in the existing `BOATS_FEATURE_STORE` and bundles them into a **Feature Group** — the Preview feature (`snowflake.ml.feature_store.FeatureGroup`) that packages multiple feature views into one versioned, single-contract object for both training (`generate_training_set(feature_group=...)`) and online inference (`read_feature_group(...)`).

Scope, per explicit instruction: **3 feature views only** — Email History, Pipeline, Listing/Form. No GA4 Behavior, no Category Prior.

Feature groups require online serving with `store_type=OnlineStoreType.POSTGRES` enabled on every source feature view — that precondition is set on all 3 below.

NOT YET RUN / VERIFIED end-to-end — built against the documented Feature Store + Feature Group API. I don't have a Python/Snowpark execution path here, only read-only SQL, so this hasn't been run. Run top to bottom and send back any errors.

In [ ]:
%%sql -r dataframe_1


In [ ]:
import snowflake.ml.feature_store as fsmod
print("FeatureGroup" in dir(fsmod))

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import (
    FeatureStore, FeatureView, FeatureGroup, Entity, CreationMode,
    OnlineConfig, OnlineStoreType,
)

session = get_active_session()
session.sql("USE ROLE DATA_SCIENTIST").collect()
session.sql("USE DATABASE AWS_DP_PROD").collect() 

# BOATS_FEATURE_STORE already exists (holds BTAPP_/BTWEB_ *_FEATURES$V* tables)
fs = FeatureStore(
    session=session,
    database="AWS_DP_PROD",
    name="BOATS_FEATURE_STORE",
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print('connected to BOATS_FEATURE_STORE')

In [ ]:
lead_entity = Entity(
    name="LEAD",
    join_keys=["LEAD_ID"],
    desc="A single inbound Lead IQ lead (email lead across boats.com / Boat Trader / YachtWorld).",
)
fs.register_entity(lead_entity)
fs.list_entities().show()

In [ ]:
# 3 feature views, scoped per instruction: no GA4 Behavior, no Category Prior.
# Each is a pass-through of already-computed columns from LEADIQ_TRAINING_SET_V5
# (a follow-up could push the raw derivation logic into feature_df directly).
#
# online_config with store_type=POSTGRES is required on every source feature
# view before they can be bundled into a FeatureGroup -- set on all 3 here.
#
# LEAD_TS in LEADIQ_TRAINING_SET_V5 is TIMESTAMP_LTZ, but the Postgres online
# store requires timestamp_col to be TIMESTAMP_NTZ -- cast once here at the
# source so all 3 feature views inherit the corrected type.
from snowflake.snowpark.functions import col, to_timestamp_ntz

src = session.table("AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V5") \
    .with_column("LEAD_TS", to_timestamp_ntz(col("LEAD_TS")))

ONLINE = OnlineConfig(enable=True, target_lag="10 minutes", store_type=OnlineStoreType.POSTGRES)
REFRESH = "60 minutes"

email_history_df = src.select(
    "LEAD_ID", "LEAD_TS",
    "N_PRIOR_LEADS", "N_DISTINCT_LISTINGS_PRIOR", "DAYS_SINCE_LAST_LEAD", "PRIOR_BOAT_OWNER",
)
pipeline_df = src.select(
    "LEAD_ID", "LEAD_TS",
    "SRC_BT_PORTAL", "SRC_BT_APP", "SRC_YW_WEB", "SRC_YW_APP",
    "SRC_BOATS_COM", "SRC_SOCIAL", "SRC_MLS", "SRC_OTHER",
)
listing_form_df = src.select(
    "LEAD_ID", "LEAD_TS",
    "LISTING_PRICE", "HAS_PHONE", "HAS_TIER", "MENTIONS_SURVEY", "MENTIONS_SEA_TRIAL",
)

fv_email_history = FeatureView(
    name="LEADIQ_EMAIL_HISTORY_FEATURES", entities=[lead_entity], feature_df=email_history_df,
    timestamp_col="LEAD_TS", refresh_freq=REFRESH, online_config=ONLINE,
    desc="Email/person prior-lead history (N_PRIOR_LEADS, N_DISTINCT_LISTINGS_PRIOR, DAYS_SINCE_LAST_LEAD, PRIOR_BOAT_OWNER).",
)
fv_pipeline = FeatureView(
    name="LEADIQ_PIPELINE_FEATURES", entities=[lead_entity], feature_df=pipeline_df,
    timestamp_col="LEAD_TS", refresh_freq=REFRESH, online_config=ONLINE,
    desc="Lead source / pipeline one-hots (SRC_BT_PORTAL ... SRC_OTHER).",
)
fv_listing_form = FeatureView(
    name="LEADIQ_LISTING_FORM_FEATURES", entities=[lead_entity], feature_df=listing_form_df,
    timestamp_col="LEAD_TS", refresh_freq=REFRESH, online_config=ONLINE,
    desc="Listing/form signals (LISTING_PRICE, HAS_PHONE, HAS_TIER, MENTIONS_SURVEY, MENTIONS_SEA_TRIAL).",
)
print('feature views defined (not yet registered):',
      fv_email_history.name, fv_pipeline.name, fv_listing_form.name)

In [ ]:
# overwrite=True makes this cell idempotent. Each auto-creates a backing
# dynamic table BOATS_FEATURE_STORE.LEADIQ_<...>_FEATURES$V1, plus a
# $V1$ONLINE table since online_config is enabled.
fv_email_history = fs.register_feature_view(fv_email_history, version="V1", overwrite=True)
fv_pipeline       = fs.register_feature_view(fv_pipeline,       version="V1", overwrite=True)
fv_listing_form   = fs.register_feature_view(fv_listing_form,   version="V1", overwrite=True)

fs.list_feature_views().filter("NAME like 'LEADIQ_%'").show()

In [ ]:

from snowflake.ml.feature_store.feature_view import StoreType

sample_lead_id = session.table("AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V5") \
    .select("LEAD_ID").limit(1).collect()[0][0]

fs.read_feature_view(fv_email_history, store_type=StoreType.ONLINE, keys=[[sample_lead_id]])

In [ ]:
# Bundle the 3 registered feature views into ONE versioned contract. This is
# what predict() should call at inference time (read_feature_group) instead
# of juggling 3 separate FeatureView lookups -- and what training should call
# (generate_training_set(feature_group=...)) to guarantee the exact same
# feature set is used both places, avoiding training-serving skew.
#
# auto_prefix=False keeps the original column names (no LEADIQ_EMAIL_HISTORY_
# FEATURES_V1_ prefix noise) since none of the 3 source FVs share a feature
# name -- only LEAD_ID (the join key) and LEAD_TS overlap, which is fine.
leadiq_fg = FeatureGroup(
    name="LEADIQ_FG",
    features=[fv_email_history, fv_pipeline, fv_listing_form],
    auto_prefix=False,
    desc="Lead IQ V1 feature bundle: email/person history + pipeline + listing/form. Single contract for training and Postgres online inference. No GA4, no Category Prior.",
)
registered_fg = fs.register_feature_group(leadiq_fg, "V1")
fs.list_feature_groups().show()


In [ ]:
fs.read_feature_group(registered_fg, keys=[[sample_lead_id]])

In [ ]:
# Spine = entity key + timestamp + label; everything else comes from the
# feature group. Source is the person-level
# target table built earlier (LEADIQ_TRAINING_V5_WITH_TARGET).
spine_df = session.table("AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_V5_WITH_TARGET").select(
    "LEAD_ID", "LEAD_TS", "LABEL"
)

training_set = fs.generate_training_set(
    spine_df=spine_df,
    feature_group=registered_fg,
    spine_timestamp_col="LEAD_TS",
    spine_label_cols=["LABEL"],
    save_as="AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V6_FS",
)
print('materialized: AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_FS_V1')
training_set.limit(5).show()

In [ ]:
# Groundwork for the inference class's predict(): a single online lookup
# against the feature group, keyed by LEAD_ID, returning all 13 features from
# the 3 bundled views in one call. This is the pattern predict() will use.
# Note: a brand-new LEAD_ID isn't queryable online until it survives one
# refresh cycle (target_lag=10 minutes here) -- this smoke test uses a
# LEAD_ID that should already be in LEADIQ_TRAINING_SET_V5.
sample_lead_id = session.table("AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V5") \
    .select("LEAD_ID").limit(1).collect()[0][0]

pdf = fs.read_feature_group(registered_fg, keys=[[sample_lead_id]])
print(pdf)

In [ ]:
chk = session.table("AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V6_FS")
n = chk.count()
n_labeled = chk.filter("LABEL = 1").count()
print(f'LEADIQ_TRAINING_SET_V6_FS: {n:,} rows, {n_labeled:,} positive ({n_labeled/n:.4%})')
# Expect ~5,910,144 rows, ~2.5% positive, and columns = LEAD_ID, LEAD_TS, LABEL
# + the 13 features from the 3 bundled views (4 email-history + 8 pipeline +
# 5 listing/form = 17, minus overlap -- no GA4's 4 columns anymore).
print(chk.columns)